<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Biomass_SSD_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rioxarray

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import rioxarray
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import xee
import geopandas as gpd

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
point = map.draw_last_feature.geometry()
point

In [ ]:
border = (
    ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
    .filterBounds(point)
)

In [ ]:
map.addLayer(border)

In [ ]:
vec = geemap.ee_to_gdf(border)
vec

In [ ]:
biomass = (
    ee.ImageCollection("NASA/ORNL/biomass_carbon_density/v1").select('agb')
    .first().rename('biomass')
)
biomass

In [ ]:
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .select('NDVI','EVI')
    .filterDate('2005','2025')
    .mean().rename('ndvi', 'evi')
)
ndvi

In [ ]:
# Load NDVI/EVI as an ImageCollection (retaining the time dimension)
ndvi_time_series = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .select('NDVI', 'EVI')
    .filterDate('2005', '2025')
)

print(f"Number of images in NDVI/EVI time series: {ndvi_time_series.size().getInfo()}")
ndvi_time_series

Now, we'll convert this `ee.ImageCollection` into an `xarray.Dataset`. The `xee` library will automatically handle the time dimension for us.

In [ ]:
# Convert the ImageCollection to an xarray.Dataset with a time dimension
# We'll use a slightly coarser scale for faster processing if there are many images
ds_ndvi_ts = xr.open_dataset(
    ndvi_time_series,
    engine='ee',
    crs='EPSG:4326',
    scale=0.005,  # Using a larger scale for faster download of time series
    geometry=border.geometry()
)

# Rename dimensions for consistency
ds_ndvi_ts = ds_ndvi_ts.rename({'lon': 'x', 'lat': 'y'})

# Clip the dataset to the border
ds_ndvi_ts_clipped = ds_ndvi_ts.rio.clip(vec.geometry, vec.crs)

display(ds_ndvi_ts_clipped)

As you can see, the `xarray.Dataset` now has a `time` dimension. We can now plot the average NDVI over time for your region of interest.

In [ ]:
plt.figure(figsize=(12, 6))
ds_ndvi_ts_clipped.NDVI.mean(dim=['x', 'y']).plot(label='Average NDVI')
plt.title('Average NDVI Over Time (2005-2025)')
plt.xlabel('Date')
plt.ylabel('Average NDVI')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('ndvitrend_map.png', dpi = 360, bbox_inches = 'tight')
plt.show()

In [ ]:
lai = (
    ee.ImageCollection("MODIS/061/MOD15A2H")
    .select('Lai_500m', 'Fpar_500m')
    .filterDate('2005', '2025')
    .mean().rename('lai', 'fpar')
)
lai

In [ ]:
temp = (
    ee.ImageCollection("MODIS/061/MYD11A2")
    .select('LST_Day_1km')
    .filterDate('2005', '2025')
    .mean().rename('temp')
)
temp

In [ ]:
pr = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_MONTHLY_V07")
    .select('precipitation')
    .filterDate('2005','2025')
    .sum().rename('pr')
)
pr

In [ ]:
lc = (
    ee.ImageCollection("MODIS/061/MCD12Q1")
    .select('LC_Type1')
    .filterDate('2005','2025')
    .first().rename('lc')
)
lc

In [ ]:
stack = ee.Image.cat([
    ndvi, lai, temp, pr, lc, biomass
])

In [ ]:
import xarray as xr

In [ ]:
from jinja2.nodes import ExprStmt
ds = xr.open_dataset(
    stack,
    engine = 'ee',
    crs = 'EPSG:4326',
    scale = 0.00108,
    geometry = border.geometry()
)
ds

In [ ]:
ds_rename = ds.rename({"lon": "x", "lat": "y"})
ds_clip = ds_rename.rio.clip(vec.geometry, vec.crs)
ds_clipped = ds_clip
df_clipped = ds_clipped.to_dataframe().dropna()
df_clipped['predicted_biomass'] = model.predict(df_clipped[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']])
xarr_clipped = df_clipped.to_xarray().sortby('x').sortby('y')

In [ ]:
df = ds.to_dataframe().dropna()
df

In [ ]:
x = df[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']]
y = df['biomass']

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)


In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators = 100,
    random_state = 42
)

model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
import numpy as np

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE:{rmse}')
r2 = r2_score(y_test, y_pred)
print(f'R2:{r2}')

In [ ]:
import xarray as xr

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
ds_clipped = ds_clip

In [ ]:
df_clipped = ds_clipped.to_dataframe().dropna()
df_clipped['predicted_biomass'] = model.predict(df_clipped[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']])

df_clipped

In [ ]:
xarr_clipped = df_clipped.to_xarray().sortby('x').sortby('y')
xarr_clipped

In [ ]:
fig, ax = plt.subplots(1,2 , figsize = (10,5))
plt.tight_layout()

xarr_clipped.biomass.plot(
    ax = ax[0],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.9}
)
ax[0].set_aspect("equal")
ax[0].set_title('biomass 2010-2011')

xarr_clipped.predicted_biomass.plot(
    ax = ax[1],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.9}
)
ax[1].set_aspect("equal")
ax[1].set_title('predicted biomass 2010-2011')
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd

In [ ]:
vec.plot()

In [ ]:
vec.crs

In [ ]:
vec.geometry

In [ ]:
ds_rename = ds.rename({"lon": "x", "lat": "y"})

In [ ]:
ds_clip = ds_rename.rio.clip(vec.geometry, vec.crs)


In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (10,5))
plt.tight_layout()

xarr_clipped.biomass.plot(
    ax = ax[0],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.9}
)
ax[0].set_aspect("equal")
ax[0].set_title('biomass 2010-2011')

xarr_clipped.predicted_biomass.plot(
    ax = ax[1],
    x = 'x',
    y = 'y',
    robust = True,
    cbar_kwargs = {'orientation': 'vertical', 'label': 'biomass Mg/ha', 'shrink': 0.9}
)
ax[1].set_aspect("equal")
ax[1].set_title('predicted biomass 2005-2025')
plt.tight_layout()
plt.savefig('biomass_map.png', dpi = 360, bbox_inches = 'tight') # Save the figure before showing it
plt.show()

In [ ]:
# Calculate the mean biomass and predicted biomass along the 'y' (latitude) axis
mean_biomass_along_x = xarr_clipped.biomass.mean(dim='y')
mean_predicted_biomass_along_x = xarr_clipped.predicted_biomass.mean(dim='y')

# Plotting trends along the 'x' (longitude) axis
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
mean_biomass_along_x.plot(label='Actual biomass', color='blue')
mean_predicted_biomass_along_x.plot(label='Predicted biomass', color='orange', linestyle='--')
plt.title('Average biomass 2005-2025')
plt.xlabel('Longitude (x)')
plt.ylabel('Average biomass Mg/ha')
plt.legend()
plt.grid(True)

# Calculate the mean biomass and predicted biomass along the 'x' (longitude) axis
mean_biomass_along_y = xarr_clipped.biomass.mean(dim='x')
mean_predicted_biomass_along_y = xarr_clipped.predicted_biomass.mean(dim='x')

# Plotting trends along the 'y' (latitude) axis
plt.subplot(1, 2, 2)
mean_biomass_along_y.plot(label='Actual biomass', color='blue')
mean_predicted_biomass_along_y.plot(label='Predicted biomass', color='orange', linestyle='--')
plt.title('Average biomass 2005-2025')
plt.xlabel('Latitude (y)')
plt.ylabel('Average biomass Mg/ha')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('average_biomass_line_chart.png') # Export the line chart
plt.savefig('bimassline_chart.png', dpi = 360, bbox_inches = 'tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Calculate the mean of each variable across x and y dimensions (and time, which is 1)
mean_values = {}
for var in ['ndvi', 'evi', 'lai', 'fpar', 'temp', 'pr', 'lc', 'biomass', 'predicted_biomass']:
    if var in xarr_clipped.data_vars:
        # Assuming 'time' dimension is always 1, so we take mean over x and y
        mean_values[var] = xarr_clipped[var].mean(dim=['x', 'y', 'time']).item()

# Create a pandas Series from the mean values
mean_series = pd.Series(mean_values)

# Create the bar plot
fig, ax = plt.subplots(figsize=(14, 7))
mean_series.plot(kind='bar', ax=ax, color='skyblue')
ax.set_title('Mean Values of All Parameters (2005-2025)')
ax.set_ylabel('Mean Value')
ax.set_xlabel('Parameter')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Get the list of all data variables except 'spatial_ref'
variables_to_plot = [var for var in xarr_clipped.data_vars if var != 'spatial_ref']

# Determine the number of rows and columns for the subplots
num_vars = len(variables_to_plot)
num_cols = 2  # Plotting trends along x and y for each variable
num_rows = (num_vars * 2 + num_cols - 1) // num_cols # Calculate rows needed to fit all plots

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows * 4))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, var in enumerate(variables_to_plot):
    # Plot trend along 'x' (longitude)
    ax_x = axes[i * 2]
    mean_along_x = xarr_clipped[var].mean(dim='y')
    mean_along_x.plot(ax=ax_x, color='blue', label=f'Mean {var}')
    ax_x.set_title(f'Average {var} along Longitude (x)')
    ax_x.set_xlabel('Longitude (x)')
    ax_x.set_ylabel(f'Average {var}')
    ax_x.grid(True)

    # Plot trend along 'y' (latitude)
    ax_y = axes[i * 2 + 1]
    mean_along_y = xarr_clipped[var].mean(dim='x')
    mean_along_y.plot(ax=ax_y, color='red', label=f'Mean {var}')
    ax_y.set_title(f'Average {var} along Latitude (y)')
    ax_y.set_xlabel('Latitude (y)')
    ax_y.set_ylabel(f'Average {var}')
    ax_y.grid(True)

# Remove any unused subplots
for j in range(num_vars * 2, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('biomassparameters_map.png', dpi = 360, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.savefig('biomass_map.png', dpi = 360, bbox_inches = 'tight')

This set of line graphs shows the average value of each parameter (NDVI, EVI, LAI, FPAR, Temperature, Precipitation, Land Cover, Biomass, and Predicted Biomass) across the longitude (x-axis) and latitude (y-axis) of the clipped region. This allows you to observe any spatial trends or variations for each individual environmental factor and biomass.

Each row represents a different parameter, with the left plot showing its average variation along longitude and the right plot showing its average variation along latitude. This approach helps in understanding the spatial distribution of each variable independently, without the visual confusion that arises from combining all parameters with different units and scales on a single plot.

#### Interpretation Note on the Combined Plot:

As mentioned, the parameters plotted above have vastly different scales and units. For example:
- **NDVI & EVI** are normalized indices, typically ranging from -1 to 1.
- **LAI & FPAR** are ratios/indices, typically ranging from 0 to 10 or 0 to 1, respectively.
- **Temp** (Land Surface Temperature) is likely in Kelvin, with values in the hundreds or thousands.
- **Pr** (Precipitation) is in millimeters, potentially with large values.
- **LC** (Land Cover Type) represents categorical values (e.g., 1 for forest, 2 for water, etc.), so its 'mean' is not directly interpretable as a continuous measurement.
- **Biomass & Predicted Biomass** are in Mg/ha, also potentially with larger values.

Therefore, this chart is primarily for comparing the presence and approximate magnitude of each parameter's average within the dataset, rather than for a direct numerical comparison between parameters. Values with much larger scales will dominate the plot's y-axis, making smaller-scaled values appear negligible.